In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, List, Optional, Any, Mapping
import sys
import copy
import yaml

# ================================================================
# CONTROL PANEL (Centralized Configuration - edit here)
# ================================================================
# 1) Static trials schedule for everyone (same schedule across all families/architectures)
DEFAULT_TRIALS_SCHEDULE: List[int] = [23] * 5

# Families to include: any subset of {"Seq2", "Darts", "Arps"}
ACTIVE_FAMILIES: List[str] = ["Seq2", "Darts", "Arps"]
ACTIVE_FAMILIES: List[str] = ["Arps"]

# Datasets to include
DATASETS_TO_RUN: List[str] = ["UNISIM_IV", "VOLVE"]
# DATASETS_TO_RUN = ["VOLVE"]

# Optional well filtering per dataset (leave {} to use all wells)
# Example: {"UNISIM_IV": ["P11", "P12"], "VOLVE": ["15/9-F-12"]}
DATASET_WELL_FILTERS: Dict[str, List[str]] = {}

# --- START OF HPO MODE CONFIGURATION SECTION ---
HPO_MODE: str = "multi-objective"  # Options: "single-objective" | "multi-objective"

HPO_PARAMS_SINGLE_OBJECTIVE: Dict[str, Any] = {
    "metric_to_optimize": "weighted_score",
    "metric_weights": {"val_smape_cum": 0.2, "val_smape_agg": 1.0},
    "lower_is_better": {"val_smape_cum": True, "val_smape_agg": True},
}

HPO_PARAMS_MULTI_OBJECTIVE: Dict[str, Any] = {
    "objectives": {"val_smape_agg": "minimize", "val_smape_cum": "minimize"},
}
# --- END OF HPO MODE CONFIGURATION SECTION ---

# Architectures to generate for each family
SEQ2_ARCHITECTURES: List[str] = ["Seq2PIN"]        # legacy: Seq2Context, Seq2Trend
DARTS_ARCHITECTURES: List[str] = ["Darts"]         # single pseudo-arch for budget mapping
ARPS_ARCHITECTURES: List[str] = ["Arps_Canonical"]

# 2) Dataset-specific split config (applied automatically per dataset)
DATASET_SPLIT_OVERRIDES: Dict[str, Dict[str, float]] = {
    "VOLVE": {"test_size": 0.55, "val_size": 0.15},
    "UNISIM_IV": {"test_size": 0.55, "val_size": 0.15},
}

# 3) Camp name:
#    - If CAMP_NAME is None, it will be auto-generated as:
#         HPO_{sum}_Lag_{lag_window}_Horizon_{horizon}
#    - If not None, it will be used as-is.
CAMP_NAME: Optional[str] = None

# ================================================================
# NEW: ONE-BUTTON METHOD CONFIG (family-aware, non-breaking)
# ================================================================
# Option A (single-family runs): set METHOD and keep METHOD_BY_FAMILY=None
METHOD: Optional[str] = None  # e.g., "PINN_PLUS_ANALYTIC"
# Options:
#   "ARPS_PURE"
#   "PINN_PURE"
#   "DARTS_PURE"
#   "PINN_PLUS_ANALYTIC"
#   "ARPS_ENSEMBLE"

# Option B (recommended for ACTIVE_FAMILIES with 2+): choose per family
METHOD_BY_FAMILY: Dict[str, str] = {
    "Seq2": "PINN_PLUS_ANALYTIC",
    "Darts": "DARTS_PURE",
    "Arps": "ARPS_PURE",
}

# How strict should compatibility be?
#   - "error": raise if missing/incompatible
#   - "fallback": use family-pure fallback if missing/incompatible
METHOD_COMPAT_POLICY: str = "error"

# ================================================================
# Job defaults (ONLY what you want to edit here)
# ================================================================
JOB_DEFAULTS_BASE_USER: Dict[str, Any] = {
    # -----------------------------
    # Core / reproducibility
    # -----------------------------
    "seed": 42,
    "architecture_name": "PLACEHOLDER",
    "feature_kind": "Normal",
    "use_known_good": False,

    # -----------------------------
    # Windowing / dataset split
    # -----------------------------
    "lag_window": 100,
    "horizon": 150,
    "patience": 50,
    "test_size": 0.55,
    "val_size": 0.15,

    # -----------------------------
    # Evaluation / reporting
    # -----------------------------
    "evaluate_by_slice": True,
    "aggregation_quantiles": [0.25, 0.5, 0.75],
    "scenario": "P50",
    "band": None,
    "show_components": False,

    # HPO-friendly default (presets may set plot=True; you can still override later)
    "plot": False,

    # Keep existing behavior stable (many parts assume agg for reporting)
    "eval_mode": "agg",

    # Keep these in base to avoid breaking current YAML expectations
    # (even if some families don't use them, existing code tolerates it)
    "aggregation_method": "reconstruct",
    "aggregation_sweep": False,
    "aggregation_candidates": [
        "reconstruct_warm_raw",
        "reconstruct_warm_ewma",
        "reconstruct_warm_holt",
        "reconstruct_warm_hp",
        "hp_hist_warm",
        "hp_raw_warm",
    ],
    "aggregation_selection_metric": "SMAPE",
}

# ================================================================
# Campaign template base (job_defaults injected per family later)
# ================================================================
CAMPAIGN_TEMPLATE_BASE: Dict[str, Any] = {
    "campaign_name": "PLACEHOLDER",
    "run_scope": {"dataset_name": "PLACEHOLDER", "wells": ["PLACEHOLDER"]},
    "hpo_params": {
        "mode": HPO_MODE,
        "trials_per_cycle_schedule": DEFAULT_TRIALS_SCHEDULE,
        "search_space_func_name": "define_search_space",
    },
    "job_defaults": JOB_DEFAULTS_BASE_USER,  # NOTE: per-family effective JD injected later
    "run_params": {"ensemble_size": 1, "max_workers": 1},
    "infra": {
        "profiles_dir": "PLACEHOLDER",
        "experiments_output_dir": "PLACEHOLDER",
        "hpo_studies_dir": "PLACEHOLDER",
        "architecture_yaml_path": "PLACEHOLDER",
        "log_level": "INFO",
    },
    "series_store": {
        "enabled": True,
        "format": "parquet",
        "compress": "zstd",
        "schema_version": 1,
        "self_heal": True,
    },
}

# --- DYNAMICALLY INJECT HPO PARAMS BASED ON MODE ---
if HPO_MODE == "multi-objective":
    CAMPAIGN_TEMPLATE_BASE["hpo_params"].update(HPO_PARAMS_MULTI_OBJECTIVE)
elif HPO_MODE == "single-objective":
    CAMPAIGN_TEMPLATE_BASE["hpo_params"].update(HPO_PARAMS_SINGLE_OBJECTIVE)
else:
    raise ValueError(f"Invalid HPO_MODE: '{HPO_MODE}'. Must be 'single-objective' or 'multi-objective'.")

# Where to write YAMLs
YAML_OUTPUT_DIR: Optional[Path] = None  # None -> resolved automatically near project root

# ================================================================
# Project path bootstrap + imports
# ================================================================
def resolve_project_root() -> Path:
    try:
        nb_pwd = Path(get_ipython().run_line_magic("pwd", "")).resolve()  # type: ignore
        project_root = Path(str(nb_pwd)).resolve()
        while project_root.name not in {"src", "notebooks"} and project_root != project_root.parent:
            project_root = project_root.parent
        if project_root.name == "notebooks":
            project_root = project_root.parent
    except Exception:
        project_root = Path.cwd().resolve()
    return project_root

PROJECT_ROOT = resolve_project_root()
SRC_PATH = PROJECT_ROOT
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

print("project_root:", PROJECT_ROOT)
print("src_path:", SRC_PATH)

from experiment_configs.hpo_campaigns.campaign_api import generate_campaign_files
try:
    from hpo.analysis_utils import print_campaign_summary
except Exception:
    print_campaign_summary = None  # optional
import campaign_api  # uses the same package as your original code

# Presets registry (needed to apply method specialization cleanly)
from forecast_pipeline.arps_offline import PIPELINE_PRESETS

# NEW: unified resolvers from src/common/common.py
from common.common import (
    FAMILY_SEQ2, FAMILY_DARTS, FAMILY_ARPS,
    summarize_methods,
    resolve_method,
    build_methods_plan_for_active_families,
    resolve_method_for_family,
    build_job_defaults_effective_for_family,
)

# ================================================================
# Helpers (pure functions, small and testable)
# ================================================================
def make_uniform_trials_map(architectures: List[str]) -> Dict[str, List[int]]:
    """Return {arch: DEFAULT_TRIALS_SCHEDULE} for each architecture."""
    return {arch: list(DEFAULT_TRIALS_SCHEDULE) for arch in architectures}

def dataset_split_overrides_for(dataset: str, base_user: Mapping[str, Any]) -> Dict[str, float]:
    """Return the test/val split overrides for a dataset (falls back to base_user)."""
    base = {"test_size": float(base_user["test_size"]), "val_size": float(base_user["val_size"])}
    return {**base, **DATASET_SPLIT_OVERRIDES.get(dataset, {})}

def auto_camp_name(job_defaults: Mapping[str, Any], trials_schedule: List[int]) -> str:
    """Build CAMP_NAME as HPO_{sum}_Lag_{lag}_Horizon_{h}."""
    lag = int(job_defaults.get("lag_window", 0))
    hz = int(job_defaults.get("horizon", 0))
    return f"HPO_{sum(trials_schedule)}_Lag_{lag}_Horizon_{hz}"

def build_base_template_resolved(project_src: Path) -> Dict[str, Any]:
    """Fill infra paths in the base template with resolved locations."""
    t = copy.deepcopy(CAMPAIGN_TEMPLATE_BASE)
    t["infra"] = {
        "profiles_dir": str(project_src / "experiment_configs" / "profiles"),
        "experiments_output_dir": str(project_src / "experiment_configs" / "results"),
        "hpo_studies_dir": str(project_src / "experiment_configs" / "studies"),
        "architecture_yaml_path": str(project_src / "experiment_configs" / "architectures_PINNs.yaml"),
        "log_level": t["infra"]["log_level"],
    }
    return t

def materialize_template_for_dataset(base_template: Dict[str, Any], dataset: str) -> Dict[str, Any]:
    """
    Create a deep-copied template customized for a single dataset:
    - Applies dataset-specific test/val sizes.
    - Keeps lag_window/horizon from JOB_DEFAULTS_BASE_USER.
    """
    t = copy.deepcopy(base_template)
    splits = dataset_split_overrides_for(dataset, JOB_DEFAULTS_BASE_USER)
    t["job_defaults"]["test_size"] = splits["test_size"]
    t["job_defaults"]["val_size"] = splits["val_size"]
    return t

def compute_yaml_output_dir(default_root: Path) -> Path:
    if YAML_OUTPUT_DIR is not None:
        return YAML_OUTPUT_DIR
    return default_root / "experiment_configs" / "hpo_campaigns"

def _effective_method_map_for_run(
    active_families: List[str],
    *,
    method: Optional[str],
    method_by_family: Optional[Mapping[str, str]],
) -> Dict[str, str]:
    """
    Normalize control panel into a per-family mapping.
    Non-breaking:
      - If METHOD_BY_FAMILY exists, it wins.
      - Else METHOD must be provided and is applied ONLY to compatible families.
        (If incompatible: handled later by policy.)
    """
    if method_by_family is not None and len(method_by_family) > 0:
        return dict(method_by_family)

    if method is None:
        raise ValueError(
            "You must set either METHOD (single-family runs) or METHOD_BY_FAMILY (multi-family runs)."
        )

    spec = resolve_method(method, allow_aliases=True)
    # Apply the same method to every active family; compatibility handled by resolver policy
    return {fam: spec.key for fam in active_families}

def _family_architectures(family: str) -> List[str]:
    if family == FAMILY_SEQ2:
        return list(SEQ2_ARCHITECTURES)
    if family == FAMILY_DARTS:
        return list(DARTS_ARCHITECTURES)
    if family == FAMILY_ARPS:
        return list(ARPS_ARCHITECTURES)
    raise ValueError(f"Unknown family={family!r}")

# ================================================================
# Orchestrator (keeps execution logic; injects per-family job_defaults)
# ================================================================
def orchestrate_campaigns(
    *,
    families: List[str],
    datasets: List[str],
    dataset_well_filters: Dict[str, List[str]] | None,
    yaml_output_dir: Path,
    camp_name: Optional[str] = None,
    method: Optional[str] = None,
    method_by_family: Optional[Mapping[str, str]] = None,
    method_policy: str = "error",
    preserve_legacy_fields: bool = True,
) -> List[Path]:
    """
    Orchestrates YAML generation across families/architectures.
    - Uniform trials schedule.
    - Dataset-specific split overrides.
    - Per-family method specialization via PIPELINE_PRESETS (non-breaking by default).
    - Per-family job_defaults injected right before YAML generation.
    """
    created_yaml_paths: List[Path] = []
    base_template = build_base_template_resolved(SRC_PATH)

    # Determine the final camp name (auto if None)
    effective_camp_name = camp_name or auto_camp_name(JOB_DEFAULTS_BASE_USER, DEFAULT_TRIALS_SCHEDULE)

    # Resolve per-family method plan
    method_map = _effective_method_map_for_run(families, method=method, method_by_family=method_by_family)

    print("\n" + summarize_methods())
    print("\nActive families:", families)
    print("Method policy:", method_policy)
    print("Method map:", method_map, "\n")

    for dataset in datasets:
        wells_by_dataset = campaign_api.build_wells_by_dataset([dataset], (dataset_well_filters or {}))

        # Dataset template (split overrides)
        dataset_template = materialize_template_for_dataset(base_template, dataset)

        # Generate per-family to allow per-family job_defaults injection safely
        for family in families:
            architectures = _family_architectures(family)
            if not architectures:
                continue

            # Resolve method spec for this family
            spec = resolve_method_for_family(
                method_by_family=method_map,
                family=family,
                policy=method_policy,
            )

            # Build per-family effective job_defaults (base_user + preset/minimal_off)
            job_defaults_effective = build_job_defaults_effective_for_family(
                base_user=dataset_template["job_defaults"],
                family=family,
                spec=spec,
                pipeline_presets=PIPELINE_PRESETS,
                preserve_legacy_fields=preserve_legacy_fields,
            )

            # Inject into a per-family template copy (avoid mutating shared objects)
            family_template = copy.deepcopy(dataset_template)
            family_template["job_defaults"] = job_defaults_effective

            trials_map = make_uniform_trials_map(architectures)

            created = generate_campaign_files(
                base_template=family_template,
                datasets=[dataset],
                wells_by_dataset=wells_by_dataset,
                family=family,
                architectures=architectures,
                output_dir=yaml_output_dir,
                trials_per_arch=trials_map,
                darts_overrides=None,
                camp_name=effective_camp_name,
            )

            print(f"[{dataset}] {family} ({spec.key}) YAMLs: {len(created)}")
            created_yaml_paths.extend(created)

    print("\nTotal YAMLs created:", len(created_yaml_paths))
    for p in created_yaml_paths[:10]:
        print(" -", p)
    return created_yaml_paths

# ================================================================
# Execute
# ================================================================
YAML_OUTPUT_DIR_RESOLVED = compute_yaml_output_dir(SRC_PATH)

created_paths = orchestrate_campaigns(
    families=ACTIVE_FAMILIES,
    datasets=DATASETS_TO_RUN,
    dataset_well_filters=DATASET_WELL_FILTERS,
    yaml_output_dir=YAML_OUTPUT_DIR_RESOLVED,
    camp_name=CAMP_NAME,
    method=METHOD,
    method_by_family=METHOD_BY_FAMILY,
    method_policy=METHOD_COMPAT_POLICY,
    preserve_legacy_fields=True,  # keep non-breaking behavior by default
)

# Optional: summarize
try:
    if print_campaign_summary:
        print_campaign_summary(created_paths)
except Exception as e:
    print("print_campaign_summary failed:", e)
